Dataset Scope
Scrape at least 100 books from a minimum of five catalog pages. Extract the following fields: title,
category, price, rating, availability, product description, UPC, number of reviews, and product URL.

In [5]:
!python -m pip install scrapy

  Using cached scrapy-2.17.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached cryptography-50.0.0-cp311-abi3-win_amd64.whl.metadata (4.3 kB)
  Using cached cssselect-1.5.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
  Using cached itemadapter-0.13.1-py3-none-any.whl.metadata (22 kB)
  Using cached itemloaders-1.4.0-py3-none-any.whl.metadata (4.2 kB)
  Using cached parsel-1.11.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached protego-0.6.2-py3-none-any.whl.metadata (6.4 kB)
  Using cached PyDispatcher-2.0.7-py3-none-any.whl.metadata (2.4 kB)
  Using cached pyopenssl-26.4.0-py3-none-any.whl.metadata (22 kB)
  Using cached queuelib-1.9.0-py3-none-any.whl.metadata (6.0 kB)
  Using cached service_identity-26.1.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached tldextract-5.3.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached twisted-26.4.0-py3-none-any.whl.metadata (15 kB)
  Using cached w3lib-2.4.1-py3-none-any.whl.metadata 


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import scrapy

Task 1 - Data Scraping
● Create a Scrapy spider that follows catalog pagination and visits each individual book page.
● Extract all required fields and export the raw records to CSV or JSON format.
● Report the total number of scraped records, missing values, and duplicate UPC values.

In [2]:
print("MY SPIDER FILE IS LOADED")
class BSpider(scrapy.Spider):
    name = 'books'
    start_urls = ['https://books.toscrape.com/catalogue/page-1.html']

    # this function is to go through every page
    def parse(self, response):
        print(response.url)
        book_links_each_page = response.css('article.product_pod h3 a::attr(href)').getall()
        print(book_links_each_page)
        for l in book_links_each_page:
            yield response.follow(l, self.parse_book) # here we are visiting each book link and storing the data in parse_book function
        next = response.css('li.next a::attr(href)').get() # this is to get the next page link
        if next:
            yield response.follow(next, self.parse)

    # the parse_book function mentioned above is written here
    # this function is mainly used to extract data of each book
    def parse_book(self,response):
        info_table = {}
        for row in response.css('table tr'):
            k = row.css('th::text').get()
            v = row.css('td::text').get()
            info_table[k] = v

        desc = response.css('#product_description + p::text').get()
        if desc is None:
            desc = 'No description available'

        category = response.css('ul.breadcrumb li a::text').getall()[2]

        rating = response.css('p.star-rating').attrib['class'].split()[-1]

        availability = response.css('p.availability::text').getall()[1].strip()

        yield {
            'title': response.css('h1::text').get(),
            'price': response.css('p.price_color::text').get(),
            'description': desc,
            'category': category,
            'rating': rating,
            'availability': availability,
            'upc': info_table['UPC'],
            "no_of_reviews": info_table['Number of reviews'],
            "product_url": response.url,
        }

print("MY SPIDER FILE IS LOADED")

MY SPIDER FILE IS LOADED
MY SPIDER FILE IS LOADED


In [6]:
!python -m pip install pandas

   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.0 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.0 MB 288.3 kB/s eta 0:00:33
   -- ------------------------------------- 0.5/10.0 MB 288.3 kB/s eta 0:00:33
   -- ------------------------------------- 0.5/10.0 MB 288.3 kB/s eta 0:00:33
   -- ------------------------------------- 0.5/10.0 MB 288.3 kB/s eta 0:00:33
   -- ------------------------------------- 0.5/10.0 M


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import pandas as pd

df = pd.read_csv(r"D:\DAU\Sem-1\ML\202618027_Devanshi_Dudhatra_DS605\202618027_Lab_01\tutorial\raw_books.csv")
print("Total Records:", len(df))
print("\nMissing Values")
print(df.isnull().sum())
print("\nDuplicate UPCs:", df["UPC"].duplicated().sum())

EmptyDataError: No columns to parse from file